In [2]:
import subprocess
import sys
from pathlib import Path

In [3]:
import os
import time

def start_infra():
    # os.chdir(project_root)
    print("Starting infra (postgres, redis)...")
    try:
        subprocess.run(["docker", "compose", "up", "-d", "postgres", "redis"], check=True)
        print("Infra started ✅")
        time.sleep(3)
        return True
    except subprocess.CalledProcessError as e:
        print("Infra failed ❌")
        print(e)
        return False

infra_ok = start_infra()
infra_ok


Starting infra (postgres, redis)...
Infra started ✅


True

In [5]:
import subprocess, os

# os.chdir(project_root)
print(subprocess.run(["docker", "compose", "ps"], capture_output=True, text=True).stdout)


NAME            IMAGE                COMMAND                   SERVICE         CREATED             STATUS                       PORTS
celery-worker   python:3.11-slim     "sh -c '\n  apt-get uâ€¦"   celery-worker   4 days ago          Up 48 minutes (healthy)      
postgres        postgres:15-alpine   "docker-entrypoint.sâ€¦"    postgres        About an hour ago   Up About an hour (healthy)   0.0.0.0:5432->5432/tcp
redis           redis:7-alpine       "docker-entrypoint.sâ€¦"    redis           7 days ago          Up 7 days (healthy)          0.0.0.0:6379->6379/tcp



In [6]:
import sys

# if str(project_root) not in sys.path:
#     sys.path.insert(0, str(project_root))

from src.config import AppConfig

cfg = AppConfig()

print("API:", cfg.api.host, cfg.api.port)
print("DB:", cfg.database.host, cfg.database.port, cfg.database.name)
print("Redis:", cfg.redis.broker_url)
print("Azure Blob container:", cfg.azure.storage.container_name)
print("ACU endpoint:", cfg.acu.endpoint)
print("ACU analyzer:", cfg.acu.analyzer_id)


API: 127.0.0.1 8000
DB: localhost 5432 credit_ocr
Redis: redis://redis:6379/0
Azure Blob container: documents
ACU endpoint: https://azure-foundry-westus-resource.services.ai.azure.com/
ACU analyzer: license_agreement_extraction_wrt_CUAD_v4_raw_normalized_singlepass


In [8]:
import subprocess
import sys
import os
import time

celery_process = None

def start_celery_worker():
    global celery_process
    if celery_process is not None:
        print("Celery already running")
        return True

    # os.chdir(project_root)
    print("Starting Celery worker...")

    celery_process = subprocess.Popen(
        [sys.executable, "-m", "celery", "-A", "src.celery_app", "worker", "--loglevel=info"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
    )

    print("Celery PID:", celery_process.pid)
    time.sleep(2)
    return True

start_celery_worker()


Starting Celery worker...
Celery PID: 49400


True

In [10]:
import subprocess
import sys
import os
import time
import requests

api_process = None
API_BASE_URL = f"http://{cfg.api.host}:{cfg.api.port}"

def start_api():
    global api_process
    if api_process is not None:
        print("API already running")
        return True

    # os.chdir(project_root)
    print("Starting API...")

    api_process = subprocess.Popen(
        [sys.executable, "run_api.py"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
    )

    print("API PID:", api_process.pid)

    # Wait for health endpoint
    for attempt in range(25):
        try:
            r = requests.get(f"{API_BASE_URL}/api/v1/health", timeout=3)
            if r.status_code == 200:
                js = r.json()
                if js.get("status") in ("healthy", "ok"):
                    print("API healthy ✅")
                    return True
                else:
                    print("API responding but not healthy yet:", js)
        except Exception:
            pass

        time.sleep(1.5)
        print(f"Waiting for API... ({attempt+1}/25)")

    print("API did not become healthy ❌")
    return False

api_ok = start_api()
api_ok


Starting API...
API PID: 48444
API healthy ✅


True

In [11]:
import requests

r = requests.get(f"{API_BASE_URL}/api/v1/health", timeout=5)
print(r.status_code)
print(r.json())

200
{'status': 'healthy', 'api_host': '127.0.0.1', 'api_port': 8000}


In [12]:
import webbrowser

web_url = f"{API_BASE_URL}/"
docs_url = f"{API_BASE_URL}/docs"

print("Web UI:", web_url)
print("Docs:", docs_url)

try:
    webbrowser.open(web_url)
except Exception as e:
    print("Could not open browser automatically:", e)


Web UI: http://127.0.0.1:8000/
Docs: http://127.0.0.1:8000/docs
